# Plotting tools {#sec-plottingtools}

Here, we introduce the more elaborate plotting tools we need to produce figures for Section @sec-results .
First, we import what we need:

In [ ]:
 #| default_exp Utils.PlottingTools

In [ ]:
 #| hide
# Imports for the nbdev development environment
import nbdev
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
 #| export
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import itertools as it
import seaborn as sns

from collections.abc import Callable

from pyDOE import lhs
import jax.numpy as jnp

from fastcore.utils import *

In [ ]:
 #| export
# Agents
from pyCRLD.Agents.Base import abase
from pyCRLD.Agents.StrategySARSA import stratSARSA
from pyCRLD.Agents.StrategyActorCritic import stratAC

from pyDDRL.Agents.DDRLBase import DDRLBase
from pyDDRL.Agents.DDRLSarsa import DDRLSarsa
from pyDDRL.Agents.DDRLActorCritic import DDRLActorCritic

# Environments
from pyDDRL.Environments.TwoArmedBandit import TwoArmedBandit

In [ ]:
 #| export
# Utilities
from pyCRLD.Utils.Helpers import *
from pyDDRL.Utils.Trajectories1D import trajectories_1D 
from pyDDRL.Utils.PhasePlot import phaseplot, _data_to_plot, _strategies, _dXisa_s
from pyCRLD.Utils import FlowPlot as fp

## Implementation

### Compare phase plots, trajectories and mean rewards for different optimism levels

In [ ]:
 #| export
## Functions to generate phase plots + trajs + mean rewards

# Obtain dX as a function of X for phase plots
# Also works when N=2 or Z=2
def obtain_dXs(aei,  # agent-environment interface
               res,  # range & resolution of strategies
               xinds:tuple,  # of indices of the phase space object to plot along the x axis
               traj_len:int,  # trajectory len
               otherXs=None,  # when N=2 or Z=2: values of X in 2nd condition
               ):
        
    l = len(res)
    dXs = np.zeros((aei.Z, l))  # will store the dXs

    for i in range(aei.Z):
        
        for j, xval in enumerate(res):  # loop over Xs
            
            if (aei.N > 1) or (aei.Z > 1): # 2 states or 2 actions
                xval_ = otherXs[j]
            else:
                xval_ = None
                
            X = obtain_Xs(aei, xinds, xval, xval_)  # create X with values along all dims
            prederror = aei.TDerror(X, norm=True)  # compute pred error
            dX = aei.step(X)[0] - X  # compute dX
            dXs[i, j] = dX[xinds[0], i, xinds[2]][0]  # dX for i,s,a of interest
            
    return dXs


def obtain_Xs(aei,  # CRLD multi-agent environment object
              xinds:tuple,  # of indices of the phase space item to plot along the x axis
              xval:float,  # the value of the phase space item to plot along the x axis
              xval_:float  # the value of X in the 2nd condition
              ): 
    
    N, C, M = aei.N, aei.Z, aei.M  # Number of agents, states, actions
    
    Xs = np.zeros((N, C, M))  # will store the Xs
    
    xi, xc, xa = xinds  # plotted agent, state, action
    xa_ = tuple(set(range(M)) - set(xa))  # other action
    
    Xs[xi, xc, xa] = xval  # setting x value
    
    # probability of taking other action
    Xs[xi,xc,xa_] = 1-Xs[xi, xc, xa]
    
    if C > 1:  # more than one state
        xc_ = tuple(set(range(C)) - set(xc))  # other state
        Xs[xi, xc_, xa] = xval_  # strategy in state 2 for action 1
        Xs[xi, xc_, xa_] = 1 - Xs[xi, xc_, xa]  # strategy in state 2 for action 2
    if N > 1:  # more than 1 agent
        xi_ = tuple(set(range(N)) - set(xi))  # other agent
        Xs[xi_, xc, xa] = xval_  # other agent's strategy for action 1
        Xs[xi_, xc, xa_] = 1 - Xs[xi_, xc, xa]  # other agent's strat for action 2
    
    return Xs

# Compute X traj and reward traj
def reward_traj(aei,  # CRLD multi-agent environment object
                X:tuple,  # strategy
                i,  # agent
                traj_len:int=500,  # trajectory length
                ): 
    
    r_traj = np.zeros((traj_len))  # will store rewards
    # Compute X trajectory
    traj, fpreached = aei.trajectory(X, traj_len) 
    # Compute reward trajectory
    for t in range(traj_len):
        r_traj[t] = aei.Ri(traj[t,...])[i]
        
    return traj, r_traj
    
# Plot phase plot, X traj and reward traj for various levels of optimism/pessimism
def compare_levels(env,  # environment
                   level_list,  # list of levels
                   gamma,  # discount factor
                   beta,  # choice intensity
                   Rmin,  # min reward perceived
                   Rmax,  # max reward perceived
                   xinds:tuple,  # which phase space axes to plot along x axes
                   res,  # specify range & resolution
                   algo='SARSA',  # specify algorithm: 'SARSA' or 'AC'
                   nrb:int=29,  # number of neurons
                   discsig:float=0.05,  # discretization sigma
                   lr:float=0.1,  # learning rate
                   traj_len:int=500,  # trajectory length
                   otherXs=None,  # if 2nd condition: values of X along this condition
                   cmap='viridis',  # colormap
                   save=False,  # save fig?
                   savename=None,  # saved figure's name
                   axes=None,  # Axes to plot into
                   ):  
   

    # Fig and Axes
    # Style
    sns.set_style("whitegrid")
    sns.set_context("paper", font_scale=1.5, rc={"lines.linewidth": 2})
    
    # Subplots: phase plot, X traj, reward traj
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(4.5 * 3.8, 3.8))
    
    # Colormap for the different lines
    cmap = mpl.colormaps[cmap]
    ext = np.linspace(0, 1, len(level_list))  # extent of colormap
    colors = cmap(tuple(ext))
    # Neutral agent is represented with a black line
    colors[int(len(level_list)/2)] = [0, 0, 0, 1]  
    
    # Zero line
    zero = np.zeros((len(res))) 

    # Loop over optimism/pessimism levels
    for j, level in enumerate(level_list):
        
        # Define agent-environment interface
        if algo=='SARSA':
            aei = DDRLSarsa(env, nr_reward_bins=nrb, discretization_sigma=discsig, 
                            learning_rates=lr, discount_factors=gamma, 
                            choice_intensities=beta, Rmin=Rmin, Rmax=Rmax,
                            method="weights", wU=level, use_prefactor=True)
        elif algo=='AC':
            aei = DDRLActorCritic(env, nr_reward_bins=nrb, discretization_sigma=discsig, 
                                  learning_rates=lr, discount_factors=gamma, 
                                  Rmin=Rmin, Rmax=Rmax,
                                  method="weights", wU=level, use_prefactor=True)
 
        # Obtain the dXs for y-axis of phase plot
        dXs = obtain_dXs(aei, res, xinds, traj_len, otherXs)
        
        # Phase plot
        ax1.plot(res, dXs[0, :], color=colors[j])
        ax1.plot(res, zero, color="grey", ls='dashed')  # plot zero line
        
        # Obtain X traj and reward traj
        X0 = np.ones((aei.N, aei.Z, aei.M)) * 0.5  # init random strategy
        X_traj, r_traj = reward_traj(aei, X0, 0, traj_len=traj_len)
        
        # X traj plot
        ax2.plot(np.arange(0, traj_len), X_traj[:,0,0,0], 
                 color=colors[j], label=r'$w_{U}=$'+str(level))
        
        # Reward plot
        ax3.plot(np.arange(0, traj_len), r_traj, 
                 color=colors[j], label=r'$w_{U}=$'+str(level))
        
    # Decorations
    ax1.set_ylabel(r"$dX$")
    ax1.set_xlabel("strategy "+r'$X$')
    ax1.set_title('phase plot')
    ax2.set_xlabel("time step "+r'$t$')
    ax2.set_ylabel("strategy "+r'$X^{0,0,0}(t)$')
    ax2.set_title('strategy over time')
    ax3.set_xlabel("time step "+r'$t$')
    ax3.set_ylabel("reward "+r'$R^{0}(t)$')
    ax3.set_title('mean reward over time')
    ax3.legend(bbox_to_anchor=(1.01, 1.01), title='weights '+r'$w_U$')
    fig.tight_layout()
    
    # Save figure
    if save == True:
        plt.savefig(savename, dpi=300, bbox_inches='tight', format='eps')


# Plot phase plot, X traj and reward traj for various levels of optimism/pessimism
def compare_gammas(env,  # environment
                   level_list,  # list of levels
                   Rmin,  # min reward perceived
                   Rmax,  # max reward perceived
                   xinds:tuple,  # which phase space axes to plot along x axes
                   res,  # specify range & resolution
                   nrb:int=29,  # number of neurons
                   discsig:float=0.05,  # discretization sigma
                   lr:float=0.1,  # learning rate
                   traj_len:int=500,  # trajectory length
                   otherXs=None,  # if 2nd condition: values of X along this condition
                   cmap='viridis',  # colormap
                   save=False,  # save fig?
                   savename=None,  # saved figure's name
                   axes=None,  # Axes to plot into
                   ):  
   

    # Fig and Axes
    # Style
    sns.set_style("whitegrid")
    sns.set_context("paper", font_scale=1.5, rc={"lines.linewidth": 2})
    
    # Subplots: phase plot, X traj, reward traj
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
    
    # Colormap for the different lines
    cmap = mpl.colormaps[cmap]
    ext = np.linspace(0, 1, len(level_list))  # extent of colormap
    colors = cmap(tuple(ext))
    
    # Zero line
    zero = np.zeros((len(res))) 

    # Loop over optimism/pessimism levels
    for j, level in enumerate(level_list):
        
        # Define agent-environment interface
        aei = DDRLActorCritic(env, nr_reward_bins=nrb, discretization_sigma=discsig, 
                              learning_rates=lr, discount_factors=level, 
                              Rmin=Rmin, Rmax=Rmax,
                              method="weights", wU=1, use_prefactor=True)
        mae = stratAC(env, learning_rates=lr, discount_factors=level, 
                      use_prefactor=True)
 
        # Obtain the dXs for y-axis of phase plot
        dXs_aei = obtain_dXs(aei, res, xinds, traj_len, otherXs)
        dXs_mae = obtain_dXs(mae, res, xinds, traj_len, otherXs)
        
        # Phase plot
        ax1.plot(res, dXs_mae[0, :], color=colors[j])
        ax1.plot(res, zero, color="grey", ls='dashed')  # plot zero line
        ax2.plot(res, dXs_aei[0, :], color=colors[j], 
                 label=r'$\gamma=$'+str(level))
        ax2.plot(res, zero, color="grey", ls='dashed')  # plot zero line
        
        
    # Decorations
    ax1.set_ylabel(r"$dX^{safe}$", fontsize=18)
    ax1.set_xlabel("strategy "+r'$X^{safe}$', fontsize=18)
    ax1.set_title('phase plots for a non-distributional agent', fontsize=16)
    ax2.set_xlabel("strategy "+r'$X$', fontsize=18)
    ax2.set_title('phase plots for a distributional agent', fontsize=16)
    ax2.legend(bbox_to_anchor=(1.01, 1.01), title='discount factor '+r'$\gamma$', fontsize=16)
    fig.tight_layout()
    
    # Save figure
    if save == True:
        plt.savefig(savename, dpi=300, bbox_inches='tight', format='eps')

### Comparison between asymmetric RL and DDRL

In [ ]:
 #| export
## Functions for comparisons between DDRL biasing methods and asymmetric RL

# Compute Q-value gap at steady state
def deltaQinf(p0,  # reward prob of arm 1 
              p1,  # reward prob of arm 2
              b):  # bias strength
    return ((p0 * b - (1 - p0)) / (p0 * b + (1 - p0))) - ((p1 * b - (1 - p1)) / (p1 * b + (1 - p1)))

# Compute prob of choosing most rewarding arm at steady state
def Xinf(dQ,  # Q-value gap
         beta):  # choice intensity
    return 1 / (1 + np.exp(beta * (-dQ)))

# Compute final strategies for quantile, expectile, weights, and asymRL methods
def compute_comparisons(p0,  # reward prob of arm 1
                        p1,  # reward prob of arm 2
                        b_ratios,  # bias strengths for asymRL
                        q_list,  # quantiles
                        e_list,  # expectiles
                        w_list,  # weights
                        gamma,  # discount factor
                        beta,  # choice intensity
                        Rmin,  # min reward perceived
                        Rmax,  # max reward perceived
                        nrb:int=29,  # number of neurons
                        discsig:float=0.05,  # discretization sigma
                        lr:float=0.1,  # learning rate
                        axes=None,  # axes to plot into
                        ):
    
    # Create 2-armed bandit
    xs = np.array([[-1, 1],
                   [-1, 1]])
    ps = np.array([[1-p0, p0],
                   [1-p1, p1]])
    env = TwoArmedBandit(disttype='discrete', xs=xs, ps=ps)
    
    # Compute asymRL Xinf for each bias strength
    b_dQs = deltaQinf(p0, p1, b_ratios)  # Q-value gaps
    b_xinf = Xinf(b_dQs, beta)  # final strategies
    
    # Compute quantile Xinf for each quantile
    q_xinf = np.zeros((len(q_list)))
    for i, q in enumerate(q_list):
        # Define agent-env interface
        aei = DDRLSarsa(env, nr_reward_bins=nrb, discretization_sigma=discsig, 
                        learning_rates=lr, discount_factors=gamma,
                        choice_intensities=beta, Rmin=Rmin, Rmax=Rmax,
                        method='quantile', tau=q, use_prefactor=False)
        # Initial strategy
        X = 0.5 * np.ones((aei.N, aei.Z, aei.M))
        # Compute strategy trajectory
        traj, fpreached = aei.trajectory(X, 150)
        # Take the last X^{0,0,0} of the trajectory
        q_xinf[i] = traj[-1, 0, 0, 0]
        
    # Compute expectile Xinf for each expectile
    e_xinf = np.zeros((len(e_list)))
    for i, e in enumerate(e_list):
        # Define agent-env interface
        aei = DDRLSarsa(env, nr_reward_bins=nrb, discretization_sigma=discsig, 
                        learning_rates=lr, discount_factors=gamma,
                        choice_intensities=beta, Rmin=Rmin, Rmax=Rmax,
                        method='expectile', tau=e, use_prefactor=False)
        # Initial strategy
        X = 0.5 * np.ones((aei.N, aei.Z, aei.M))
        # Compute strategy trajectory
        traj, fpreached = aei.trajectory(X, 150)
        # Take the last X^{0,0,0} of the trajectory
        e_xinf[i] = traj[-1, 0, 0, 0]
        
    # Compute weights Xinf for each upper weight value
    w_xinf = np.zeros((len(w_list)))
    for i, w in enumerate(w_list):
        # Define agent-env interface
        aei = DDRLSarsa(env, nr_reward_bins=nrb, discretization_sigma=discsig, 
                        learning_rates=lr, discount_factors=gamma,
                        choice_intensities=beta, Rmin=Rmin, Rmax=Rmax,
                        method='weights', wU=w, use_prefactor=False)
        # Initial strategy
        X = 0.5 * np.ones((aei.N, aei.Z, aei.M))
        # Compute strategy trajectory
        traj, fpreached = aei.trajectory(X, 150)
        # Take the last X^{0,0,0} of the trajectory
        w_xinf[i] = traj[-1, 0, 0, 0]
        
    return b_xinf, q_xinf, e_xinf, w_xinf

# Plot the final strategies
def plot_comparisons(b_ratios, q_list, e_list, w_list, b_xinf, q_xinf, 
                     e_xinf, w_xinf, scarcity="poor", save=False,
                     savename=None):
    
    # Style
    sns.set_style("whitegrid")
    sns.set_context("paper", font_scale=1.5, rc={"lines.linewidth": 2})
    # Subplots: quantile, expectile, weights
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(4.5 * 3.8, 4.5))
    axes = [ax1, ax2, ax3]
    # Twin axes and their properties
    ax11 = ax1.twiny()
    ax22 = ax2.twiny()
    ax33 = ax3.twiny()
    twin_axes = [ax11, ax22, ax33]
    ddrl_lists = [q_list, e_list, w_list]
    ddrl_xinfs = [q_xinf, e_xinf, w_xinf]
    colors = ['red', 'green', 'orange']
    labels = ['quantile', 'expectile', 'DDRL']
    xlabels = [r'$\tau$', r'$\tau$', 'weight '+r'$w_U$']
    
    # to find the bias strength of the neutral asymRL agent
    ind = np.sum(b_ratios < 1)
    
    # The plots
    for i, ax in enumerate(axes):
        ax.set_xscale('log')  # log scale for x-axis
        # Plot Xinfs for asym RL
        ax.plot(b_ratios, b_xinf, '-+', mew=3, ms=8, color='blue', label="asym RL")
        # Plot neutral line
        ax.plot(b_ratios, np.repeat(b_xinf[ind], len(b_ratios)), '--', color='gray')
        # Plot Xinfs for DDRL method
        if i==1:
            axes[i].plot(b_ratios, ddrl_xinfs[i], '-o', color=colors[i],
                         label=labels[i])
        else:
            twin_axes[i].plot(ddrl_lists[i], ddrl_xinfs[i], '-o', color=colors[i],
                              label=labels[i])
        
   
    # Instructions for specific axes
    ax33.set_xscale('log')
    
    # Decorations
    for i, ax in enumerate(axes):
        if scarcity=="poor":
            ax.legend(loc='upper left')
        else: 
            ax.legend(loc='upper right')
        ax.set_xlabel('bias strength '+r'$b$', color="blue")  
        ax.xaxis.label.set_color('blue')
        ax.tick_params(axis='x', colors='blue')
        ax.spines['bottom'].set_color('blue')
        ax.spines['top'].set_color(colors[i])
        twin_axes[i].set_xlabel(xlabels[i], color=colors[i])
        twin_axes[i].xaxis.label.set_color(colors[i])
        twin_axes[i].tick_params(axis='x', colors=colors[i])
        twin_axes[i].spines['bottom'].set_color('blue')
        twin_axes[i].spines['top'].set_color(colors[i])
    
    # Decorations for specific axes    
    ax1.set_ylabel('final strategy '+r'$X_{\infty}$')
    # ax22.set_xticks(e_list)
    
    if scarcity=="poor":
        ax11.legend(loc="center left")
        ax33.legend(loc='center left')
    else:
        ax11.legend(loc='center right')
        ax33.legend(loc='center right')
    
    ax1.set_title("asymmetric RL vs. quantiles")
    ax2.set_title("asymmetric RL vs. expectiles")
    ax3.set_title("asymmetric RL vs. DDRL in a poor environment")
    
    fig.tight_layout()
    
    # Save figure
    if save == True:
        plt.savefig(savename, dpi=300, bbox_inches='tight', format='eps')

### Plot 3 flowplots with associated trajectories

In [ ]:
 #|export
## Functions to plot 3 flowplots and associated trajectories

# Plot the flowplots together
def plot_3flowplots(aeis,  # list of aeis
                    x:tuple,  # which phase space axes to plot along x axes
                    y:tuple,  # which phase space axes to plot along y axes
                    NrRandom:int=4,  # number of random strategies to average over
                    Xinits=None,  # initial Xs to plot trajectories
                    decorate=False,  # whether to decorate the axes or not
                    titles=None,  # list of titles for 3 plots
                    xlabel=None,  # x-label
                    ylabel=None,  # y-label
                    cols=None,  # list of colors for trajectories
                    cmap='viridis',  # Colormap
                    axes=None,  # Axes to plot into
                    ):
    
    # Style
    sns.set_style("white")
    sns.set_context("paper", font_scale=1.25, rc={"lines.linewidth": 2})
    
    # Plot the flow plot
    for i, ax in enumerate(axes):
        aei = aeis[i]
        fp.plot_strategy_flow(aei, x, y, flowarrow_points = np.linspace(0.01 ,0.99, 9), 
                              use_RPEarrows=False, NrRandom=NrRandom, cmap=cmap, axes=ax)
        
        # Plot trajectory if initial strategies are specified
        if Xinits != None:
            trajs = []  # trajectories
            fprs = []  # are fixed points reached?
            for _, Xinit in enumerate(Xinits):
                trj, fpr = aei.trajectory(Xinit, Tmax=12000, tolerance=1e-6)
                trajs.append(trj)
                fprs.append(fpr)
                
            fp.plot_trajectories(trajs, x, y, fprs=fprs, axes=ax, cols=cols)
    
    # Add titles, xlabels and ylabels to subplots if needed
    if decorate==True:
        for i, ax in enumerate(axes): 
            ax.set_xlabel(xlabel)
            ax.set_title(titles[i])
        
        axes[0].set_ylabel(ylabel)

### Plot separatrix on flowplots

In [ ]:
 #|export
def compile_strategy(p0c:float,  # cooperation probability of agent zero
                     p1c:float):  # cooperation probability of agent one
    strat = np.zeros((2, 1, 2))
    strat[0, 0, 0] = p0c
    strat[1, 0, 0] = p1c
    strat[0, 0, 1] = 1 - p0c
    strat[1, 0, 1] = 1 - p1c
    return strat                         

def plot_separatrix(aei, ax, x, y):    
    # Add saddle node
    # by reversing the dynamics from two agents with identical strategies
    o = [0.5, 0.5]; o = compile_strategy(*o)
    for _ in range(1000): o, _ = aei.reverse_step(o)
    # Add separatrix
    o = o[:, 0, 0] - np.array([0, 0.0001])
    o1 = compile_strategy(*o)
    o2 = compile_strategy(*o[::-1])
    sep1=[]; sep2=[]
    for _ in range(1000): o1, _ = aei.reverse_step(o1); sep1.append(o1)
    for _ in range(1000): o2, _ = aei.reverse_step(o2); sep2.append(o2)
    fp.plot_trajectories([sep1, sep2], x=x, y=y, cols=['dimgray'], lws=[1],
                         lss=['--'], alphas=[0.95], axes=ax)

### Generate stability landscape

In [ ]:
 #|export
def fill_containers(env, Xrisk, Xsafe, discountfactors, weights):
    risky_optimal_data_container = np.zeros((discountfactors.size, weights.size, 2))
    risky_stable_container = np.zeros((discountfactors.size, weights.size, 2))
    safe_stable_container = np.zeros((discountfactors.size, weights.size, 2))
    for i, dcf in enumerate(discountfactors):
        for j, w in enumerate(weights):
            aei = DDRLActorCritic(env, nr_reward_bins=29, discretization_sigma=0.05, 
                                  learning_rates=0.1, discount_factors=dcf, 
                                  Rmin=0, Rmax=1,
                                  method="weights", wU=w, use_prefactor=True)
            Visa_risk = aei.Visa(Xrisk)
            dX_risk = aei.step(Xrisk)[0] - Xrisk
            risky_stable_container[i, j, :] = dX_risk[0, :, 0] <= 0
            ag = 0; a = 1; s = 2
            Vis_risk = jnp.einsum(Visa_risk, [ag, s, a], Xrisk, [ag, s, a], [ag, s])
            Visa_safe = aei.Visa(Xsafe)
            dX_safe = aei.step(Xsafe)[0] - Xsafe
            safe_stable_container[i, j, :] = dX_safe[0, :, 0] >= 0
            Vis_safe = jnp.einsum(Visa_safe, [ag, s, a], Xsafe, [ag, s, a], [ag, s])
            risky_optimal_data_container[i, j, :] = Vis_risk[0,:] > Vis_safe[0,:]
            print("gamma = "+str(dcf)+", w = "+str(w)+" done")
    safe_stable_container = safe_stable_container * 2
    indicator_stable = safe_stable_container + risky_stable_container
    return risky_optimal_data_container, indicator_stable

In [ ]:
 #| hide
import nbdev; nbdev.nbdev_export()